# Execution file for ASSUAGE

This notebook provides an example workflow that incorporates all parts of the ASSUAGE methodology.

There are three components:
1. Creating the ground truth dataset.
2. Building the surrogate model, including curation and training of the machine-learning layer.
3. Integrating the surrogate with PyMOS within an optimisation workflow.

All parts of the methodology are modular, and users are encouraged to install the code as an editable folder to adapt it to their needs.

This notebook contains example ``full`` and ``reduced`` models to demonstrate the workflow. These are located in the ``exampleTemplates`` directory.

A design is represented as:
[var1, var2, var3]


The full model evaluates the fitness as:
$ \text{true\_fitness} = 2\,\text{var1}^2 + \frac{3}{2}\,\text{var2} - 4\,\text{var3}^{3/2} $

The reduced model does not have access to this fitness directly. However, it can interact with the input variables and construct engineered variables:
var1³, var2·var3

The surrogate model developed within this pipeline attempts to approximate the true fitness value using the available inputs:
[var1, var2, var3, var1$^3$, var2·var3]

In this example, both the full and reduced models run quickly (approximately 1 second per run each). In typical applications, the methodology is intended for scenarios where the full model is significantly more computationally expensive than the reduced-order model.



In [ ]:
# Import any additional packages used in this jupyter notebook

import os
import numpy as np


## Ground truth dataset creation

The surrogate model contains an ML layer, whihc is curated and trained offline, before being incorporated into the optimisation pipeline. To accurately train this layer, ground truth data is needed, which is obtained from runs of the full model. 

This code initiates running the full model to create the ground truth data. The full model is a priori assumed to be expensive, so the user can set the number of parallel jobs as requires, and restart jobs if needed.

A simulation is finished if the file ``endedSim.txt` is located in the simulation directory.

**User-set parameters:**
- `numNewRuns`: total number of new runs (new datapoints) to create
- `numCoresPerSim`: maximum number of CPU cores used per simulation  
- `numCores`: total number of CPU cores available for generating the ground-truth dataset

The code will run ``numCores`` / ``numCoresPerSim`` simulations simultaneously, until a total of `numNewRuns` of new simulations have been completed.


In [ ]:
## Add this line in to remove all previous files.
#os.system(f"rm -rf groundTruth surrogateCreation")

numNewRuns = 0
numCoresPerSim = 1
numCores = 10

fullModelTemplate = os.path.join(os.getcwd(), "exampleTemplates","fullModelTemplate")

# Set parameter bounds. The length of these lists determine the number of parameter.
lowerBounds = [0] * 3
upperBounds = [1] * 3
paramNames = [f"var{i}" for i in range(3)]
assert len(lowerBounds) == len(upperBounds), "Upper and lower bound lists must have the same length."


In [ ]:

try:
    from encoding import preprocess_parameters as preprocess_func
except:
    print("No preprocess_parameters function found in encoding file")
    preprocess_func = None

try:
    from encoding import parameter_to_model
except:
    print("No parameter_to_model function found in encoding file. This is a necessary function!!")

## Create ground truth data set
from ASSUAGE.create_ground_truth import start_new_runs
start_new_runs(numNewRuns, fullModelTemplate, lowerBounds, upperBounds, numCoresPerSim, numCores, parameter_to_model, preprocess_func);


Starting new set of runs with first id 250 . Number of new runs 0


### Extract data for the surrogate model from the ground truth dataset

This step requires two functions from `encoding.py`: `extract_surrogate_inputs` and `extract_fitness`.

`extract_surrogate_inputs` runs a reduced model for each full-model simulation and uses the outputs of the reduced-order model as the inputs for the ML surrogate layer.

`extract_fitness` extracts the fitness value as computed by the ground truth model.

Both outputs are stored as CSV files in `surrogateCreation`.  
The next stage of the pipeline builds an ML layer that correlates these inputs with the corresponding outputs.


In [ ]:
from ASSUAGE.create_ground_truth import extractData

try: 
    from encoding import extract_surrogate_inputs, extract_fitness
except:
    print("Critical surrogate input and output functions not found. Check these are defined in encoding.py")

extractData(extract_surrogate_inputs, extract_fitness);

Starting to extract data from run0
Successfully extracted data from run0
Starting to extract data from run1
Successfully extracted data from run1
Starting to extract data from run10
Successfully extracted data from run10
Starting to extract data from run100
Successfully extracted data from run100
Starting to extract data from run101
Successfully extracted data from run101
Starting to extract data from run102
Successfully extracted data from run102
Starting to extract data from run103
Successfully extracted data from run103
Starting to extract data from run104
Successfully extracted data from run104
Starting to extract data from run105
Successfully extracted data from run105
Starting to extract data from run106
Successfully extracted data from run106
Starting to extract data from run107
Successfully extracted data from run107
Starting to extract data from run108
Successfully extracted data from run108
Starting to extract data from run109
Successfully extracted data from run109
Starting 

In [ ]:
from ASSUAGE.surrogate_model.data_exploration import data_exploration

explorer = data_exploration("surrogateCreation/trainingInput.csv","surrogateCreation/trainingOutput.csv")
explorer.correlation_matrix()
print("Correlation matrix created.")

explorer.feature_histograms()
print("Feature histograms created.")

missing_report = explorer.missing_value_report()
print("Missing values report:\n", missing_report.head())

reduced_data = explorer.explanatory_dimension(0.95) # PCA dimensionality reduction to preserve 95% variance
print("Reduced dataset shape:", reduced_data.shape)

# 5. PCA variance table
variance_table = explorer.explain_variance_table()
print("PCA variance table created.")

explorer.pairplot(sample=150)
print("Pairplot created (without hue).")

explorer.pairplot(sample=150, hue=0)
print("Pairplot created (with hue from output).")

outlier_mask = explorer.outlier_detection(method="zscore", thresh=3.0)
print("Outlier count:", outlier_mask.sum())

importances = explorer.feature_importance_proxy(target_column=0)
print("Top feature importances:\n", importances.head())

print("\nAll demonstration outputs written to 'surrogateCreation/'.")

Surrogate data loaded correctly.
Total NaN values: 0
Removed constant columns: []
Visualisations will use scaled data.
Correlation matrix created.
Feature histograms created.
Missing values report:
    feature  n_missing  pct_missing
0        0          0          0.0
1        1          0          0.0
2        2          0          0.0
3        3          0          0.0
4        4          0          0.0
Reduced dataset shape: (250, 4)
PCA variance table created.
Pairplot created (without hue).
Pairplot created (with hue from output).
Outlier count: 13
Top feature importances:
 2    0.904963
1    0.031458
0    0.024679
3    0.021679
4    0.017222
dtype: float64

All demonstration outputs written to 'surrogateCreation/'.


In [ ]:
from ASSUAGE.surrogate_model.ml_models import mlModels


modeler = mlModels(
    input_data="surrogateCreation/trainingInput.csv",
    output_data="surrogateCreation/trainingOutput.csv",
    data_label="example",
    output_dir="surrogateCreation",
    pca=True,            # no PCA to keep things simple
    search_type="random",
    n_iter=5,            # small n_iter for quick demo
    n_jobs=10,            # 1 to avoid parallel issues in demo
    seed=42,
    best_model_cutoff=0.95
)

results = modeler.evaluate_holdout(test_size=0.1, inner_cv=5, create_plots=True)
#results = modeler.evaluate_nested_cv(outer_splits=5, inner_splits=5, create_plots=True)

print("\Best model recorded in the object:")
print(" best_model_name:", modeler.best_model_name)
print(" best_model_score:", modeler.best_model_score)
print(" best_model_info:")
print(modeler.best_model_info)

# --- Train best pipeline (auto-selected) and save with pickle (the class uses pickle) ---
print("\nRunning train_best_pipeline() (uses recorded best model)...")
pipeline = modeler.train_best_pipeline(cv=5)  # will save pipeline to output_dir

print("\nPipeline saved to:", modeler.best_pipeline_path)
os.system(f"cp {modeler.best_pipeline_path} surrogateCreation/best_pipeline.pkl")



[holdout] Searching for model: Linear regression
[holdout] Linear regression done in 1.6s, R2=0.9723
[holdout] Searching for model: Ridge
[holdout] Ridge done in 0.0s, R2=0.9724
[holdout] Searching for model: Lasso
[holdout] Lasso done in 0.1s, R2=0.9724
[holdout] Searching for model: Bayesian Ridge
[holdout] Bayesian Ridge done in 0.1s, R2=0.9724
[holdout] Searching for model: Decision Tree
[holdout] Decision Tree done in 0.1s, R2=0.8884
[holdout] Searching for model: Random Forest
[holdout] Random Forest done in 1.3s, R2=0.9573
[holdout] Searching for model: Gradient Boosting
[holdout] Gradient Boosting done in 0.5s, R2=0.9823
[holdout] Searching for model: SVR
[holdout] SVR done in 0.1s, R2=0.9766
[holdout] Searching for model: MLP
[holdout] MLP done in 0.2s, R2=0.9981
[holdout] Models exceeding cutoff:
   Linear regression: R2=0.9723
   Ridge: R2=0.9724
   Lasso: R2=0.9724
   Bayesian Ridge: R2=0.9724
   Random Forest: R2=0.9573
   Gradient Boosting: R2=0.9823
   SVR: R2=0.9766
   

Further data visualisation of the most important features in creating the model.

In [ ]:

from ASSUAGE.surrogate_model.ml_plotter import feature_importance
import pickle
import pandas as pd
# --- Load the saved pipeline via pickle and predict on first 5 rows ---
with open(modeler.best_pipeline_path, "rb") as fh:
    loaded_pipe = pickle.load(fh)

sample_X = pd.read_csv("surrogateCreation/trainingInput.csv", header=None).iloc[:5].to_numpy()
preds = loaded_pipe.predict(sample_X)
print("\nPredictions on first 5 samples (loaded pipeline):")
print(preds)

feature_importance(loaded_pipe, output_dir=modeler.output_dir, X=modeler.X, y= modeler.y)

print("All outputs written into 'surrogateCreation/example' folder.")


Predictions on first 5 samples (loaded pipeline):
[ 0.69958082 -0.41254889  0.33192335 -2.25755144  0.12480452]
[feature_importance] Computing permutation importance (n_repeats=30). This may be slow.
[feature_importance] Saved plot to surrogateCreation/example/feature_importances_1764674810.png
[feature_importance] Saved CSV to surrogateCreation/example/feature_importances_1764674810.csv  (reason=permutation_importance (n_repeats=30); mapping=mapped_from_pca_components)
All outputs written into 'surrogateCreation/example' folder.


### Optimisation 
The final part of the methodology 

In [ ]:
## Optional code to test the functionality of the fitness run

from ASSUAGE.optimisation.optimisation_fitness import Fitness
f = Fitness(parameter_to_model, extract_surrogate_inputs, surrogate_template_folder="exampleTemplates/reducedModelTemplate",
        bounds=  (lowerBounds, upperBounds), parameter_names=paramNames, preprocess_func=preprocess_func,
        simulation_folder="SimulationTest", clean_dir=False)
print(f.fitness(np.random.random(size=(24)), id=0).fitness)
print(f.fitness(np.random.random(size=(24)), id=1).fitness)

## Remove previous files
os.system(f"rm -rf SimulationTest fitness.csv discarded_parameters.csv");


Starting fitness evaluation at id = 0
-0.33730649265363666
Starting fitness evaluation at id = 1
0.13392423030729883


In [ ]:

from ASSUAGE.optimisation.run_optimisation import runOptimisation

optimiser = runOptimisation(parameter_to_model=parameter_to_model, extract_surrogate_func=extract_surrogate_inputs, 
                     surrogate_template_folder="exampleTemplates/reducedModelTemplate", surrogate_file="surrogateCreation/best_pipeline.pkl",
                     bounds=  (lowerBounds, upperBounds), parameter_names=paramNames, preprocess_func=preprocess_func)

values = optimiser.run_opt_realisation(seed=42, budget=100, n_jobs=10, simulation_folder="run_optimisation_test", 
                                log_folder="optimisation_logs", clean_dir=True)

optimiser.run_ground_truth(simulation_folder="optimisation_logs", 
                    fullModelTemplate=fullModelTemplate, 
                    parameter_to_model=parameter_to_model,
                    #values = values,
                    preprocess_func=preprocess_func,
                    seed=42)



Starting fitness evaluation at id = 0
Starting fitness evaluation at id = 1
Starting fitness evaluation at id = 2
Starting fitness evaluation at id = 3
Starting fitness evaluation at id = 4
Starting fitness evaluation at id = 5
Starting fitness evaluation at id = 6
Starting fitness evaluation at id = 7
Starting fitness evaluation at id = 8
Starting fitness evaluation at id = 9


Exception: Extracting results have been requested for a pending future. This should not happen.